In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score,balanced_accuracy_score, recall_score

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


### Load the embeddings for the C.S.sylv sulcal region

In [2]:
base_model_path = '/neurospin/dico/data/deep_folding/current/models/Champollion_V0'
list_model_path = [f'{base_model_path}_trained_on_UKB40/SC-sylv_right/11-36-10_85_0/ukb40_random_epoch80_embeddings/full_embeddings.csv',
                   f'{base_model_path}/SC-sylv_right/11-43-38_3/ukb40_random_epoch100_embeddings/full_embeddings.csv',
                   f'{base_model_path}/SC-sylv_right/13-19-08_28/ukb40_random_epoch80_embeddings/full_embeddings.csv',
                   f'{base_model_path}/SC-sylv_right/13-35-41_0/ukb40_random_epoch100_embeddings/full_embeddings.csv',
                   f'{base_model_path}/SC-sylv_right/13-35-41_1/ukb40_random_epoch100_embeddings/full_embeddings.csv',
                   f'{base_model_path}/SC-sylv_right/13-35-41_2/ukb40_random_epoch100_embeddings/full_embeddings.csv'

]
ukb_embeddings = pd.read_csv(list_model_path[1], index_col=0) 
print(ukb_embeddings.shape)
ukb_embeddings.head()

(42433, 256)


,dim1,dim2,dim3,dim4,dim5,dim6,dim7,dim8,dim9,dim10,...,dim247,dim248,dim249,dim250,dim251,dim252,dim253,dim254,dim255,dim256
ID,,,,,,,,,,,,,,,,,,,,,
sub-1000021,-37.629620,6.583590,29.279840,-16.611984,-30.081190,-24.476210,11.042302,-6.784241,19.132568,10.150197,...,-2.848078,11.085325,-12.354747,-76.037210,-8.468971,-29.602938,-24.906027,-16.344492,7.703534,-1.050617
sub-1000325,11.934219,11.355382,-13.496794,-38.996407,14.297873,-5.811631,6.174108,-45.331050,9.762536,44.107750,...,4.212949,6.075599,-36.018467,-134.833560,26.447592,-35.490543,4.641970,-4.289551,-58.478560,-31.515226
sub-1000458,-25.394463,4.755465,-22.513622,-25.682335,-67.619896,-18.027046,-30.610876,51.657852,-18.840885,-7.459018,...,-9.459454,-0.912658,32.695583,12.509913,10.790437,-25.991050,-39.407890,-13.558734,47.675068,-0.354762
sub-1000575,-25.367662,-23.005732,-30.035063,36.237038,-103.579190,-9.643719,4.125421,34.603440,19.479185,-13.243917,...,20.220034,-5.049628,4.961305,-25.468930,25.991018,3.052118,-73.673270,6.855866,30.889896,43.402153
sub-1000606,-35.599197,-2.058082,26.222765,-3.495081,-49.495464,-3.061627,-14.236676,-3.899254,-15.054834,50.399190,...,8.652889,27.369568,34.453293,-73.026400,40.409150,28.420736,-51.241203,17.027500,-0.707319,-20.734170


### Reduce dimension (hope to remove the noise) with a PCA

In [3]:
n_components=200

pca = PCA(n_components=n_components)
pca.fit(ukb_embeddings)
print(pca.explained_variance_ratio_)
(np.cumsum(pca.explained_variance_ratio_) < 0.999).sum()

[1.66038777e-01 1.21728397e-01 1.11881942e-01 1.10596436e-01
 9.33253624e-02 8.97115051e-02 8.18137776e-02 5.80327224e-02
 4.79934755e-02 3.26441361e-02 2.28046101e-02 1.54360804e-02
 1.22888179e-02 8.17237026e-03 6.51910661e-03 4.70459698e-03
 2.91733933e-03 2.51396977e-03 1.86247073e-03 1.71999419e-03
 1.24140786e-03 9.03856299e-04 7.62301366e-04 6.92234163e-04
 4.66106133e-04 3.64479422e-04 2.90182431e-04 2.70263813e-04
 2.47258036e-04 2.17806035e-04 1.75298918e-04 1.55898483e-04
 1.37857504e-04 1.14012511e-04 9.23282183e-05 8.46075754e-05
 7.84219467e-05 6.71849118e-05 6.64322744e-05 5.93484064e-05
 5.19775289e-05 4.97552706e-05 4.41281447e-05 4.23864947e-05
 3.58957990e-05 3.22198644e-05 3.03193128e-05 2.87940961e-05
 2.66898852e-05 2.45674981e-05 2.24218463e-05 2.11197973e-05
 2.00533934e-05 1.98746867e-05 1.83175732e-05 1.71301852e-05
 1.58129435e-05 1.48397500e-05 1.31452588e-05 1.26457638e-05
 1.23216034e-05 1.19727891e-05 1.05609293e-05 9.92661844e-06
 9.53377867e-06 8.882302

36

In [4]:
ukb_pca_bdd = pca.transform(ukb_embeddings)

In [23]:
#scaler = StandardScaler()
#scaler.fit(ukb_embeddings)
#ukb_scl_bdd = scaler.transform(ukb_embeddings)
#ukb_scl_bdd

In [5]:
QCdf = pd.read_csv('/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/CSLabel/interrupted_CS_QC.csv')
QCdf.Note.unique()

array(['OK', 'C.S. in the right of the mask, partially out',
       'C.S. in the left of the mask, partially out',
       'Ambiguous, not a clear interruption', 'Issue with the white mesh',
       'Very short C.S.', 'No interruption'], dtype=object)

In [6]:
interrupted = QCdf[QCdf.Note=="OK"].ID.to_list()

ambiguous =  QCdf[QCdf.Note=='Ambiguous, not a clear interruption'].ID.to_list()
ambiguous = ambiguous + [
'sub-1310920',
'sub-2863742',
'sub-1911266',
'sub-4217758',
'sub-5222070',
'sub-3794487',
'sub-1420697',
'sub-1425827',
'sub-3891499',
'sub-3572724',
'sub-1053493',
'sub-4875056',
'sub-1527779', 
'sub-4397096', 
'sub-3109923',
'sub-4860959',
'sub-2425148',
'sub-2077194',
'sub-1649070',
'sub-2005939',
'sub-2889389',
'sub-4520944',
'sub-4281714',
'sub-3692612', 
'sub-2379487', 
'sub-5137278',
'sub-4571621',
'sub-2461041',
'sub-1140601',
'sub-3439492',
'sub-3816101',
'sub-1452398',
'sub-5474299',
'sub-5080160',
'sub-3834564',
'sub-1372315',
'sub-5426258',
'sub-3313248',
'sub-3538950',
'sub-2642697',
'sub-5493039',
'sub-3272797',
'sub-3902778',
'sub-2929118', 
'sub-4257283',
'sub-5522199',
'sub-3604986',
'sub-5185480',
'sub-5623262',
'sub-5317805',
'sub-3297125',
'sub-2489075',
'sub-1647006',
]

not_interrupted = [
'sub-3943435',
'sub-4854284',
'sub-4558487',
'sub-1977658',
'sub-5161517',
'sub-2077690',
'sub-5937161',
'sub-4326429',
'sub-1377158',
'sub-3462570',
'sub-4342190',
'sub-4816666',
'sub-3884683',
'sub-1103646',
'sub-1167379',
'sub-1190643',
'sub-1273718',
'sub-1286007',
'sub-1298876',
'sub-1352284',
'sub-1398736',
'sub-1422413',
'sub-1465129',
'sub-1597706',
'sub-1701563',
'sub-1734788',
'sub-1979982',
'sub-1996092',
'sub-2036033',
'sub-2097565',
'sub-2118136',
'sub-2141551',
'sub-2193253',
'sub-2207793',
'sub-2228486',
'sub-2284024',
'sub-2337820',
'sub-2349203',
'sub-2389411',
'sub-2420937',
'sub-2427515',
'sub-2538754',
'sub-2583027',
'sub-2592717',
'sub-2733674',
'sub-2741815',
'sub-2792782',
'sub-2802489',
'sub-2816262',
'sub-2833426',
'sub-2834970',
'sub-2837393',
'sub-2946274',
'sub-2957401',
'sub-2968297',
'sub-2970418',
'sub-3008660',
'sub-3009279',
'sub-3013938',
'sub-3227039',
'sub-3234836',
'sub-3264612',
'sub-3333294',
'sub-3334219',
'sub-3379262',
'sub-3388080',
'sub-3388306',
'sub-3401499',
'sub-3453064',
'sub-3525594',
'sub-3529189',
'sub-3541105',
'sub-3603191',
'sub-3627711',
'sub-3670173',
'sub-3693543',
'sub-3721299',
'sub-3722413',
'sub-3765466',
'sub-3936967',
'sub-3992259',
'sub-3994474',
'sub-4016129',
'sub-4027732',
'sub-4116944',
'sub-4411765',
'sub-4420611',
'sub-4428393',
'sub-4491384',
'sub-4519441',
'sub-4536778',
'sub-4727825',
'sub-4741296',
'sub-4755899',
'sub-4787289',
'sub-4791977',
'sub-4805119',
'sub-4805237',
'sub-4834994',
'sub-4868991',
'sub-5027399',
'sub-5054716',
'sub-5082433',
'sub-5117110',
'sub-5123219',
'sub-5147403',
'sub-5217534',
'sub-5237880',
'sub-5292898',
'sub-5293703',
'sub-5319071',
'sub-5430535',
'sub-5437419',
'sub-5486726',
'sub-5561142',
'sub-5578922',
'sub-5581707',
'sub-5605784',
'sub-5643778',
'sub-5649675',
'sub-5686761',
'sub-5723111',
'sub-5729132',
'sub-5749108',
'sub-5754849',
'sub-5836983',
'sub-5864979',
'sub-5910947',
'sub-5966409',
'sub-5998652',
'sub-5357627',
'sub-2204575',
'sub-2839753',
'sub-5335727',
'sub-5782466',
'sub-4520082',
'sub-1004170',
'sub-4158073', 
'sub-5684893',
'sub-4359496',
'sub-2040983',
'sub-5575777',
'sub-1116938',
'sub-4189639',
'sub-4507392',
'sub-5085553',
'sub-5457081',
'sub-4831688',
'sub-3976041',
'sub-4057189',
'sub-4202490',
'sub-4844615',
'sub-4747425',
'sub-1008582',
'sub-4039492',
'sub-2969851', 
'sub-2830945', 
'sub-1711798', 
'sub-5120758',
'sub-4949491', 
'sub-4772821', 
'sub-4450785', 
'sub-1110891',
'sub-2920350',
'sub-5217882',
'sub-5747063',
'sub-5040811',
'sub-3143577',
'sub-1779953',
'sub-2112770',
'sub-2327257',
'sub-2406636',
'sub-3745349',
'sub-2968941', 
'sub-2611694',
'sub-2676227',
'sub-1164936',
'sub-4794448',
'sub-5711287',
'sub-1502163',
'sub-1546206',
'sub-4710850',
'sub-3735109',
'sub-2800235',
'sub-5115775',
'sub-3746225',
'sub-1416140',
'sub-5907102',
'sub-5989851',
'sub-3974444',
'sub-2212279',
'sub-4899126',
'sub-4518664', 
'sub-5828387',
'sub-1385217',
'sub-2442019',
'sub-1273391',
'sub-2302082', 
'sub-2814161',
'sub-3785229',
'sub-4945539',
'sub-4479433',
'sub-1930115',
'sub-1218580',
'sub-4880356',
'sub-5159616',
'sub-1809920',
'sub-4089007',
'sub-2127220',
'sub-4479596',
'sub-3276505',
'sub-1367494',
'sub-2480789',
'sub-3889542', 
'sub-5912354',
'sub-4593980',
'sub-1219530',
'sub-5751516',
'sub-4804513',
'sub-4006078',
'sub-1828965',
'sub-4025167',
'sub-2598789',
'sub-3344616',
'sub-4791222',
'sub-5224500',
'sub-1100724',
'sub-4690601',
'sub-3791953',
'sub-4661668',
'sub-4164110',
'sub-4431586',
'sub-1109550',
'sub-4606013',
'sub-2899381', 
'sub-2234711', 
'sub-3624620', 
'sub-2980957', 
'sub-4932490',
'sub-2258124', 
'sub-4119595', 
'sub-4404054', 
'sub-2368134', 
'sub-2496737',
'sub-3519440', 
'sub-4698380', 
'sub-1874775', 
'sub-3797977',
'sub-1340306',
'sub-1691707',
'sub-3314602',
'sub-5412919',
'sub-1529050'
] 

In [7]:
"""
Problem with:
[
'sub-3716267',
'sub-5417598',
'sub-5472164',
'sub-5894417',
'sub-1703355',
'sub-4271898',
]
"""

"\nProblem with:\n[\n'sub-3716267',\n'sub-5417598',\n'sub-5472164',\n'sub-5894417',\n'sub-1703355',\n'sub-4271898',\n]\n"

In [8]:
X = ukb_embeddings.loc[interrupted + not_interrupted]
y = [1 for i in range(len(interrupted))] + [0 for i in range(len(not_interrupted))]
X_pca = pca.transform(X)
len(interrupted), len(not_interrupted)

(207, 253)

In [9]:
print(len(set(interrupted)), len(interrupted), '\n')
print(len(set(not_interrupted)), len(not_interrupted), '\n')
print(set([x for x in interrupted if interrupted.count(x) > 1]))
print(set([x for x in not_interrupted if not_interrupted.count(x) > 1]))
print((set.intersection(set(interrupted), set(not_interrupted))))
print((set.intersection(set(interrupted), set(ambiguous))))
print((set.intersection(set(ambiguous), set(not_interrupted))))

my_labelled_df = pd.DataFrame({"ID":interrupted + not_interrupted, "Interruption":y})
#my_labelled_df.to_csv('/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/CSLabel/interrupted_CS_labelled.csv', index=False)

ambiguous_df = pd.DataFrame({"ID":ambiguous})
#ambiguous_df.to_csv('/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/CSLabel/ambiguous_CS_antoine.csv', index=False)

207 207 

253 253 

set()
set()
set()
set()
set()


In [10]:
df_interrupted_julien = pd.read_csv('/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/CSLabel/interrupted_CS_julien.csv')
df_ambiguous_julien = pd.read_csv('/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/CSLabel/ambiguous_CS_julien.csv')


print('No interruption (me) VS interruption (Julien):', df_interrupted_julien.ID.isin((my_labelled_df[my_labelled_df.Interruption==0]).ID).sum(), '\n')
print('Interruption for both of us:',df_interrupted_julien.ID.isin((my_labelled_df[my_labelled_df.Interruption==1]).ID).sum(), '\n')
print("Interruption (Julien) I don't have as interruption:",len((df_interrupted_julien[~df_interrupted_julien.ID.isin((my_labelled_df[my_labelled_df.Interruption==1]).ID)])),'\n')
print("Length of interruption Julien's list:", len(df_interrupted_julien), '\n')

print('No interruption (me) VS ambiguous (Julien):',df_ambiguous_julien.ID.isin((my_labelled_df[my_labelled_df.Interruption==0]).ID).sum(), '\n')
print('Interruption (me) VS ambiguous (Julien):',df_ambiguous_julien.ID.isin((my_labelled_df[my_labelled_df.Interruption==1]).ID).sum(), '\n')

print('Ambiguous (me) VS interruption (Julien):',df_interrupted_julien.ID.isin(ambiguous).sum(), '\n')
print('Ambiguous (me) VS ambiguous (Julien):',df_ambiguous_julien.ID.isin(ambiguous).sum(), '\n')
print("Length of ambiguous Julien's list:", len(df_ambiguous_julien), '\n')

No interruption (me) VS interruption (Julien): 0 

Interruption for both of us: 33 

Interruption (Julien) I don't have as interruption: 19 

Length of interruption Julien's list: 52 

No interruption (me) VS ambiguous (Julien): 0 

Interruption (me) VS ambiguous (Julien): 0 

Ambiguous (me) VS interruption (Julien): 15 

Ambiguous (me) VS ambiguous (Julien): 5 

Length of ambiguous Julien's list: 35 



In [11]:
X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca, y, test_size=0.33, random_state=42)

#### linear SVC model

In [12]:
model = SVC(kernel='linear', probability=True,
            random_state=42,
            C=0.001, class_weight='balanced')

For model comparison, we calculate the ROC AUC (wihout PCA) with the previous 

In [13]:
for path_i in list_model_path:
    ukb_embeddings = pd.read_csv(path_i, index_col=0) 
    X = ukb_embeddings.loc[interrupted + not_interrupted]
    outputs = {}
    val_pred = cross_val_predict(model, X, y, cv=5)
    auc = roc_auc_score(y, val_pred)
    outputs['labels_pred'] = val_pred
    outputs['auc'] = auc
    outputs['balanced_accuracy_score'] = balanced_accuracy_score(y, val_pred)

    print(path_i)
    print('ROC AUC (cv=5):', "{:.3f}".format(outputs['auc']), '\n')

/neurospin/dico/data/deep_folding/current/models/Champollion_V0_trained_on_UKB40/SC-sylv_right/11-36-10_85_0/ukb40_random_epoch80_embeddings/full_embeddings.csv
ROC AUC (cv=5): 0.756 

/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/11-43-38_3/ukb40_random_epoch100_embeddings/full_embeddings.csv
ROC AUC (cv=5): 0.839 

/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/13-19-08_28/ukb40_random_epoch80_embeddings/full_embeddings.csv
ROC AUC (cv=5): 0.813 

/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/13-35-41_0/ukb40_random_epoch100_embeddings/full_embeddings.csv
ROC AUC (cv=5): 0.751 

/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/13-35-41_1/ukb40_random_epoch100_embeddings/full_embeddings.csv
ROC AUC (cv=5): 0.810 

/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/13-35-41_2/ukb40_random_epoch100_embeddings/full_embeddings.csv
ROC AUC (

In [14]:
outputs = {}
val_pred = cross_val_predict(model, X_pca, y, cv=5)
auc = roc_auc_score(y, val_pred)
outputs['labels_pred'] = val_pred
outputs['auc'] = auc
outputs['balanced_accuracy_score'] = balanced_accuracy_score(y, val_pred)

print('ROC AUC (cv=5):', "{:.3f}".format(outputs['auc']), '\n')
print('Balanced accuracys score (cv=5):', "{:.3f}".format(outputs['balanced_accuracy_score']))

ROC AUC (cv=5): 0.839 

Balanced accuracys score (cv=5): 0.839


In [15]:
model.fit(X_train_pca, y_train)

print('Recall (test):', "{:.3f}".format(recall_score(y_test, model.predict(X_test_pca))), '\n')
print('ROC AUC (test):', "{:.3f}".format(roc_auc_score(y_test ,model.predict_proba(X_test_pca)[:,1])), '\n')
print('Balanced accuracy score (test):', "{:.3f}".format(balanced_accuracy_score(y_test, model.predict(X_test_pca))), '\n')
model.fit(X_pca, y)

Recall (test): 0.849 

ROC AUC (test): 0.913 

Balanced accuracy score (test): 0.836 



SVC(C=0.001, class_weight='balanced', kernel='linear', probability=True,
    random_state=42)

In [29]:
prediction = pd.DataFrame({"ID" : list(ukb_embeddings.index),
              "Pred" : model.predict_proba(ukb_pca_bdd)[:,1]})
#prediction.to_csv('/volatile/ad279118/UKB/CentralSulcus/interruption_pred.csv', index=False)
prediction

,ID,Pred
0,sub-1000021,0.024508
1,sub-1000325,0.005363
2,sub-1000458,0.031206
3,sub-1000575,0.037652
4,sub-1000606,0.001041
...,...,...
42428,sub-6023847,0.041374
42429,sub-6024038,0.012000
42430,sub-6024150,0.035936
42431,sub-6024379,0.001833


In [17]:
print('Maximum probability of prediction among the interrupted C.S. :',"{:.3f}".format(prediction[prediction["ID"].isin(interrupted)].Pred.max()), '\n')
print('Mean probability of prediction among the interrupted C.S. :', "{:.3f}".format(prediction[prediction["ID"].isin(interrupted)].Pred.mean()), '\n')
prediction[prediction['ID']=='sub-2036033']

Maximum probability of prediction among the interrupted C.S. : 1.000 

Mean probability of prediction among the interrupted C.S. : 0.816 



,ID,Pred
8695,sub-2036033,0.105518


In [18]:
((prediction[~(prediction["ID"].isin(interrupted))]).sort_values(by="Pred")[-5:].ID).to_list()

['sub-4447456', 'sub-2774064', 'sub-2625764', 'sub-5474299', 'sub-4589882']

To compare to: 

category.tsv

| category_id | title                                         | availability | group_type | descript                                                                 | notes                                                                   |
|-------------|-----------------------------------------------|--------------|------------|--------------------------------------------------------------------------|-------------------------------------------------------------------------|
| 136         | Mental health                                 | 0            | 1          | Results of the on-line mental health self-assessment questionnaire issued in 2016. |                                                                         |
| 137         | Mental distress                               | 0            | 1          | Mental distress reported within the on-line mental health questionnaire. |                                                                         |
| 138         | Depression                                    | 0            | 1          | Depression reported within the on-line mental health questionnaire.      |                                                                         |
| 139         | Mania                                         | 0            | 1          | Mania reported within the on-line mental health questionnaire.           |                                                                         |
| 140         | Anxiety                                       | 0            | 1          | Anxiety reported within the on-line mental health questionnaire.         |                                                                         |
| 141         | Addictions                                    | 0            | 1          | Addictions reported within the on-line mental health questionnaire.      |                                                                         |
| 142         | Alcohol use                                   | 0            | 1          | Alcohol use reported within the on-line mental health questionnaire.     |                                                                         |
| 143         | Cannabis use                                  | 0            | 1          | Cannabis use reported within the on-line mental health questionnaire.    |                                                                         |
| 144         | Unusual and psychotic experiences             | 0            | 1          | Unusual and psychotic experiences reported within the on-line mental health questionnaire. |                                                                         |
| 145         | Traumatic events                              | 0            | 1          | Traumatic events reported within the on-line mental health questionnaire. |                                                                         |
| 146         | Self-harm behaviours                          | 0            | 1          | Self-harm behaviours reported within the on-line mental health questionnaire. |                                                                         |
| 147         | Happiness and subjective well-being           | 0            | 1          | Happiness and subjective well-being reported within the on-line mental health questionnaire. |                                                                         |
| 2415        | Pregnancy, childbirth and the puerperium      | 0            | 1          | First reported occurrences of conditions falling within the ICD10 classification Chapter XV Pregnancy, childbirth and the puerperium. |                                                                         |


encoding.tsv

| encoding_id | title                    | availability | coded_as | structure | num_members | descript                                                                   |
|-------------|--------------------------|--------------|----------|-----------|-------------|----------------------------------------------------------------------------|
| 100694      | Bipolar type             | 0            | 11       | 1         | 2           | Type of bipolar episode                                                    |
| 1405        | Depression substances     | 0            | 11       | 1         | 4           | Substances taken to potentially alleviate depressive symptoms.              |
| 1406        | Depression therapies      | 0            | 11       | 1         | 3           | Non-drug therapies aimed at alleviating depressive symptoms                  |
| 1908        | Antidepressant medications| 0            | 11       | 1         | 9           | Antidepressant medications                                                  |
| 3006        | Depression frequency      | 0            | 11       | 1         | 5           | Frequency of current depression symptoms with option of prefer not to answer |
| 100695      | Depressive episode        | 0            | 11       | 1         | 6           | Type of depressive episode(s)                                               |

field.tsv

| field_id | title                                         | availability | stability | private | value_type | base_type | item_type | strata | instanced | arrayed | sexed | units | main_category | encoding_id | instance_id | instance_min | instance_max | array_min | array_max | num_participants | item_count | showcase_order | cost_do | cost_on | cost_sc |
|----------|-----------------------------------------------|--------------|-----------|--------|------------|-----------|-----------|--------|-----------|---------|-------|-------|---------------|-------------|-------------|--------------|--------------|-----------|-----------|-----------------|------------|----------------|---------|---------|---------|
| 46       | Hand grip strength (left)      | 0            | 2         | 0      | 11         | 0         | 0         | 0      | 1         | 0       | 0     | Kg    | 100019        | 0           | 2           | 0            | 3            | 0         | 0         | Left grip strength. An issue has been identified with a small amount of grip strength data (~F46~ and ~F47~) collected in Cheadle during the first repeat visit in 2013 (Instance 1). | 2012-01-05T00:00:00 | 2023-10-01T00:00:00 | 499213           | 599524     | 1              | 1       | 1       | 1       |
| 47       | Hand grip strength (right)     | 0            | 2         | 0      | 11         | 0         | 0         | 0      | 1         | 0       | 0     | Kg    | 100019        | 0           | 2           | 0            | 3            | 0         | 0         | Right grip strength. An issue has been identified with a small amount of grip strength data (~F46~ and ~F47~) collected in Cheadle during the first repeat visit in 2013 (Instance 1). | 2012-01-05T00:00:00 | 2023-10-01T00:00:00 | 499291           | 599607     | 1.5            | 1       | 1       | 1       |
| 1707     | Handedness (chirality/laterality)             | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100033     | 100430      | 2            | 0            | 2         | 0         | 501463          | 533456     | 5              | 1       | 1       | 1       |
| 1920     | Mood swings                                   | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501480          | 604063     | 1              | 1       | 1       | 1       |
| 1930     | Miserableness                                 | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501480          | 604063     | 2              | 1       | 1       | 1       |
| 1940     | Irritability                                  | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501480          | 604063     | 3              | 1       | 1       | 1       |
| 1950     | Sensitivity / hurt feelings                   | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501479          | 604062     | 4              | 1       | 1       | 1       |
| 1960     | Fed-up feelings                               | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501479          | 604062     | 5              | 1       | 1       | 1       |
| 1970     | Nervous feelings                              | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501479          | 604062     | 6              | 1       | 1       | 1       |
| 1980     | Worrier / anxious feelings                    | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501478          | 604061     | 7              | 1       | 1       | 1       |
| 1990     | Tense / 'highly strung'                       | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501478          | 604061     | 8              | 1       | 1       | 1       |
| 2000     | Worry too long after embarrassment            | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501478          | 604061     | 9              | 1       | 1       | 1       |
| 2010     | Suffer from 'nerves'                          | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501478          | 604061     | 10             | 1       | 1       | 1       |
| 2020     | Loneliness, isolation                         | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501477          | 604060     | 11             | 1       | 1       | 1       |
| 2030     | Guilty feelings                               | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501477          | 604060     | 12             | 1       | 1       | 1       |
| 2040     | Risk taking                                   | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501477          | 604060     | 13             | 1       | 1       | 1       |
| 2050     | Frequency of depressed mood in last 2 weeks   | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100484      | 2            | 0            | 3         | 0         | 501477          | 604060     | 20             | 1       | 1       | 1       |
| 2060     | Frequency of unenthusiasm / disinterest in last 2 weeks | 0 | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100484      | 2            | 0            | 3         | 0         | 501476          | 604059     | 21             | 1       | 1       | 1       |
| 2070     | Frequency of tenseness / restlessness in last 2 weeks | 0 | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100484      | 2            | 0            | 3         | 0         | 501475          | 604058     | 22             | 1       | 1       | 1       |
| 2080     | Frequency of tiredness / lethargy in last 2 weeks | 0 | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100484      | 2            | 0            | 3         | 0         | 501474          | 604057     | 23             | 1       | 1       | 1       |
| 2090     | Seen doctor (GP) for nerves, anxiety, tension or depression | 0 | 0 | 0 | 21 | 11 | 0 | 0 | 1 | 0 | 0 | | 100060 | 100349 | 2 | 0 | 3 | 0 | 0 | 501473 | 604056 | 24 | 1 | 1 | 1 |
| 2100     | Seen a psychiatrist for nerves, anxiety, tension or depression | 0 | 0 | 0 | 21 | 11 | 0 | 0 | 1 | 0 | 0 | | 100060 | 100349 | 2 | 0 | 3 | 0 | 0 | 501473 | 604056 | 25 | 1 | 1 | 1 |



#### Second approach: Euclidian distance in the reduced latent space

In [18]:
from scipy.spatial import distance

In [521]:
list_dist = [distance.euclidean(pca.transform(ukb_embeddings.loc['sub-3791185'].to_numpy().reshape(1,-1)), ukb_pca_bdd[i]) for i in range(len(ukb_pca_bdd))]
df_dist = pd.DataFrame({"ID":list(ukb_embeddings.index), "Dist":list_dist})

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr

In [522]:
sample_dist = ((df_dist[~(df_dist["ID"].isin(interrupted))]).sort_values(by='Dist').iloc[22000:22025].ID).to_list()

### Visualization with Anatomist

In [19]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims

existing QApplication: 0
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-ad279118'


create qapp
global modules: /casa/host/build/share/anatomist-5.2/python_plugins
home   modules: /casa/home/.anatomist/python_plugins
done
Starting Anatomist.....
config file : /casa/home/.anatomist/config/settings.cfg
PyAnatomist Module present
PythonLauncher::runModules()
loading module simple_controls
loading module save_resampled
loading module selection
loading module bsa_proba
loading module modelGraphs
loading module profilewindow
loading module ana_image_math
loading module paletteViewer
loading module foldsplit
loading module anacontrolmenu
loading module gradientpalette
loading module palettecontrols
loading module meshsplit
loading module volumepalettes
loading module gltf_io
loading module infowindow
loading module histogram
loading module measure
loading module statsplotwindow
loading module valuesplotwindow
all python modules loaded
Anatomist started.


In [20]:
dataset = 'UkBioBank40'
region = "S.C.-sylv."
side = "R"

mm_skeleton_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}crops'

In [23]:
sample = ((prediction[~(prediction["ID"].isin(interrupted+ambiguous+not_interrupted))]).sort_values(by="Pred", ascending=False)[25:50].ID).to_list()

In [26]:
volume_files = []

for subject_id in sample:
    volume_path = f"{mm_skeleton_path}/{subject_id}_cropped_skeleton.nii.gz"

    if os.path.isfile(volume_path):
        vol = aims.read(volume_path)
        volume_files.append(vol)
    else:
        print(f"{volume_path} is not a correct path, or the .nii.gz doesn't exist")

block = a.createWindowsBlock(5) # 10 columns
dic_windows = {}

for i, vol in enumerate(volume_files):
    dic_windows[f'a_vol{i}'] = a.toAObject(vol)
    #dic_windows[f'a_vol{i}'].setPalette(absoluteMode=True)
    dic_windows[f'rvol{i}'] = a.fusionObjects(objects=[dic_windows[f'a_vol{i}']], method='VolumeRenderingFusionMethod')
    dic_windows[f'rvol{i}'].releaseAppRef()
    dic_windows[f'wvr{i}'] = a.createWindow('3D', block=block) #geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
    dic_windows[f'wvr{i}'].addObjects(dic_windows[f'rvol{i}'])

no position could be read at 558, 229
no position could be read at 350, 90
no position could be read at 319, 89
no position could be read at 160, 71
no position could be read at 410, 110


# Analysis

In [26]:
Birth_Weight = pd.read_csv('/volatile/ad279118/UKB/CentralSulcus/BirthWeight.csv')
print(Birth_Weight.shape, '\n')
print(Birth_Weight.columns,'\n')
print('Nb of Nan for "Birth weight | Instance 0":', Birth_Weight["participant.p20022_i0"].isna().sum())
print('Nb of Nan for "Birth weight | Instance 1":',Birth_Weight["participant.p20022_i1"].isna().sum())
print('Nb of Nan for "Birth weight | Instance 2":',Birth_Weight["participant.p20022_i2"].isna().sum())

Birth_Weight["Birth_Weight"] = Birth_Weight["participant.p20022_i0"]
Birth_Weight = Birth_Weight.drop(["participant.p20022_i0", "participant.p20022_i1", "participant.p20022_i2"], axis=1)
Birth_Weight["ID"] = Birth_Weight["0"].apply(lambda x : 'sub-'+str(x))
Birth_Weight = Birth_Weight.drop("0", axis=1)
Birth_Weight.head()

(42402, 4) 

Index(['0', 'participant.p20022_i0', 'participant.p20022_i1',
       'participant.p20022_i2'],
      dtype='object') 

Nb of Nan for "Birth weight | Instance 0": 16990
Nb of Nan for "Birth weight | Instance 1": 37970
Nb of Nan for "Birth weight | Instance 2": 38633


,Birth_Weight,ID
0,NaN,sub-1000021
1,0.94,sub-1000325
2,NaN,sub-1000458
3,2.38,sub-1000575
4,3.40,sub-1000606


In [48]:
merged = pd.merge(left=prediction, right=Birth_Weight, left_on='ID', right_on='ID', how='inner')
#merged = merged.drop('Participant ID', axis=1)
interrupted_with_BW = merged[merged.ID.isin(interrupted)].dropna()
not_interrupted_with_BW = merged[merged.ID.isin((prediction.sort_values(by='Pred')).iloc[1000:30000,:].ID)].dropna()
print(interrupted_with_BW[["Pred", "Birth_Weight"]].mean(axis=0),'\n')
print(not_interrupted_with_BW[["Pred", "Birth_Weight"]].mean(axis=0),'\n')

Pred            0.785347
Birth_Weight    3.394342
dtype: float64 

Pred            0.015546
Birth_Weight    3.353757
dtype: float64 

